In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader

from langchain_classic.retrievers.parent_document_retriever import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [3]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model_name= "gpt-3.5-turbo", max_tokens=500)

In [4]:
# Carregar o PDF
pdf_link = "Clean Architecture.pdf"

loader = PyPDFLoader(pdf_link, extract_images=False)

pages = loader.load_and_split()
len(pages)


381

In [5]:
# Splitters para separação do documento em chunks

child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 4000,
    chunk_overlap = 200,
    length_function = len,
    add_start_index = True
)

In [6]:
# Storages -> Chroma para o recuperar os documentos relevantes e 
# InMemory fica o documento pai pois só precisa para enviar junto para o modelo

memoryStore = InMemoryStore()
vectorstore = Chroma(embedding_function=embeddings, persist_directory='db/childVectorDB')  

In [7]:
parent_document_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=memoryStore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

batch_size = 20 # Número de documentos a serem adicionados por vez
for i in range(0, len(pages), batch_size):
    batch = pages[i:i + batch_size]
    parent_document_retriever.add_documents(batch, ids=None)
    print(f"Adicionados {len(batch)} documentos. Total processado: {i + len(batch)}/{len(pages)}")

print("Todos os documentos foram adicionados.")

Adicionados 20 documentos. Total processado: 20/381
Adicionados 20 documentos. Total processado: 40/381
Adicionados 20 documentos. Total processado: 60/381
Adicionados 20 documentos. Total processado: 80/381
Adicionados 20 documentos. Total processado: 100/381
Adicionados 20 documentos. Total processado: 120/381
Adicionados 20 documentos. Total processado: 140/381
Adicionados 20 documentos. Total processado: 160/381
Adicionados 20 documentos. Total processado: 180/381
Adicionados 20 documentos. Total processado: 200/381
Adicionados 20 documentos. Total processado: 220/381
Adicionados 20 documentos. Total processado: 240/381
Adicionados 20 documentos. Total processado: 260/381
Adicionados 20 documentos. Total processado: 280/381
Adicionados 20 documentos. Total processado: 300/381
Adicionados 20 documentos. Total processado: 320/381
Adicionados 20 documentos. Total processado: 340/381
Adicionados 20 documentos. Total processado: 360/381
Adicionados 20 documentos. Total processado: 380/3